# Feature selection: pruning the corners feature set

**Purpose.** Decide which feature families to keep in the Q1 final model.

**Core idea.** A feature is worth keeping only if removing it makes validation
MAE worse. Tree models can produce many features but most gradient-boosted setups
are robust to redundancy — the question is whether each *family* of features
adds signal on top of the others, not whether each individual feature does.

**Approach.**
1. Build the full candidate feature set (10 families, ~70 columns).
2. Train baseline LightGBM (L1 objective, early stopping) on home and away.
3. Score each family with three importance measures:
   - **Built-in tree gain**: training-loss reduction attributable to each feature.
     Free; biased toward continuous / high-cardinality features; collinear features
     split gain among themselves so a useful family can look unimportant.
   - **Single-feature permutation on val**: shuffle one column, measure ΔMAE.
     Directly tied to generalisation. Same collinearity weakness as gain.
   - **Grouped permutation on val**: shuffle a whole family at once.
     Resolves collinearity — answers *"is this family redundant given the rest?"*
4. Drop families with ΔMAE_grouped ≈ 0 (or negative).

## Load data and build full candidate feature set

The full set spans 10 families: mixed rolling, venue-split rolling, EWM rolling,
opponent-adjusted rolling, league-shrunk strength, league-normalised "above" deltas,
matchup expected, matchup edges (mixed and venue), and competition priors.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor, early_stopping

train_raw = pd.read_parquet('train.parquet')
betting   = pd.read_parquet('betting.parquet')
train_raw['date_time']   = pd.to_datetime(train_raw['date_time'])
betting['date_time']     = pd.to_datetime(betting['date_time'])
betting_matches = betting.drop_duplicates('match_id').copy()

model_data = train_raw[train_raw['competition_id'].isin(betting_matches['competition_id'])].copy()
model_data = model_data.sort_values('date_time').reset_index(drop=True)
model_data['total_corners'] = model_data['home_corners'] + model_data['away_corners']

print('model_data:', model_data.shape)

model_data: (11198, 14)


In [2]:
home_rows = model_data[['match_id', 'date_time', 'home_team_id',
                        'home_corners', 'away_corners']].rename(
    columns={'home_team_id': 'team_id',
             'home_corners': 'corners_for', 'away_corners': 'corners_against'})
home_rows['is_home'] = 1
away_rows = model_data[['match_id', 'date_time', 'away_team_id',
                        'away_corners', 'home_corners']].rename(
    columns={'away_team_id': 'team_id',
             'away_corners': 'corners_for', 'home_corners': 'corners_against'})
away_rows['is_home'] = 0
team_games = pd.concat([home_rows, away_rows], ignore_index=True)
team_games = team_games.sort_values(['team_id', 'date_time', 'match_id']).reset_index(drop=True)

# mixed rolling (l5, l10)
for w in [5, 10]:
    team_games[f'cf_l{w}'] = team_games.groupby('team_id')['corners_for'].transform(
        lambda s: s.shift(1).rolling(w, min_periods=1).mean())
    team_games[f'ca_l{w}'] = team_games.groupby('team_id')['corners_against'].transform(
        lambda s: s.shift(1).rolling(w, min_periods=1).mean())

# venue-split rolling
team_games = team_games.sort_values(['team_id', 'is_home', 'date_time', 'match_id']).reset_index(drop=True)
for w in [5, 10]:
    team_games[f'cf_venue_l{w}'] = team_games.groupby(['team_id', 'is_home'])['corners_for'].transform(
        lambda s: s.shift(1).rolling(w, min_periods=1).mean())
    team_games[f'ca_venue_l{w}'] = team_games.groupby(['team_id', 'is_home'])['corners_against'].transform(
        lambda s: s.shift(1).rolling(w, min_periods=1).mean())
team_games = team_games.sort_values(['team_id', 'date_time', 'match_id']).reset_index(drop=True)

# EWM rolling (recent matches weighted more)
for hl in [5, 10]:
    team_games[f'cf_ewm_hl{hl}'] = team_games.groupby('team_id')['corners_for'].transform(
        lambda s: s.shift(1).ewm(halflife=hl, ignore_na=True).mean())
    team_games[f'ca_ewm_hl{hl}'] = team_games.groupby('team_id')['corners_against'].transform(
        lambda s: s.shift(1).ewm(halflife=hl, ignore_na=True).mean())

# opponent-adjusted: subtract opp's pre-match conceding/scoring rate, then roll
opp = team_games[['match_id', 'team_id', 'ca_l10', 'cf_l10']].rename(
    columns={'team_id': 'opp_id', 'ca_l10': 'opp_ca', 'cf_l10': 'opp_cf'})
team_games = team_games.merge(opp, on='match_id')
team_games = team_games[team_games['team_id'] != team_games['opp_id']].drop('opp_id', axis=1)
team_games = team_games.sort_values(['team_id', 'date_time', 'match_id']).reset_index(drop=True)

g = team_games['corners_for'].mean()
team_games['opp_ca'] = team_games['opp_ca'].fillna(g)
team_games['opp_cf'] = team_games['opp_cf'].fillna(g)
team_games['cf_adj']  = team_games['corners_for']     - team_games['opp_ca']
team_games['ca_adj']  = team_games['corners_against'] - team_games['opp_cf']
team_games['cf_adj_l10'] = team_games.groupby('team_id')['cf_adj'].transform(
    lambda s: s.shift(1).rolling(10, min_periods=1).mean())
team_games['ca_adj_l10'] = team_games.groupby('team_id')['ca_adj'].transform(
    lambda s: s.shift(1).rolling(10, min_periods=1).mean())
team_games['n_prior'] = team_games.groupby('team_id').cumcount()

print('team_games rolling done:', team_games.shape)

team_games rolling done: (22396, 25)


In [3]:
# competition priors and shrinkage strength
hist = model_data.sort_values(['competition_id', 'season_id', 'date_time']).copy()
for col in ['home_corners', 'away_corners', 'total_corners']:
    hist[f'comp_{col}_prev'] = hist.groupby(['competition_id', 'season_id'])[col].transform(
        lambda x: x.shift(1).expanding().mean())
hist['comp_home_adv_prev'] = hist['comp_home_corners_prev'] - hist['comp_away_corners_prev']
comp_cols = ['comp_home_corners_prev', 'comp_away_corners_prev',
             'comp_total_corners_prev', 'comp_home_adv_prev']

team_games = team_games.merge(hist[['match_id'] + comp_cols], on='match_id', how='left')
team_games['league_for'] = np.where(team_games['is_home'] == 1,
                                     team_games['comp_home_corners_prev'],
                                     team_games['comp_away_corners_prev'])
team_games['league_against'] = np.where(team_games['is_home'] == 1,
                                         team_games['comp_away_corners_prev'],
                                         team_games['comp_home_corners_prev'])

K = 6
for w in [5, 10]:
    n = np.minimum(team_games['n_prior'], w).astype(float)
    w_team, w_lg = n / (n + K), K / (n + K)
    cf = team_games[f'cf_l{w}'].fillna(team_games['league_for'])
    ca = team_games[f'ca_l{w}'].fillna(team_games['league_against'])
    team_games[f'attack_shrunk_l{w}']  = w_team * cf + w_lg * team_games['league_for']
    team_games[f'defense_shrunk_l{w}'] = w_team * ca + w_lg * team_games['league_against']
    team_games[f'attack_above_l{w}']   = team_games[f'attack_shrunk_l{w}']  - team_games['league_for']
    team_games[f'defense_above_l{w}']  = team_games[f'defense_shrunk_l{w}'] - team_games['league_against']

print('team_games full:', team_games.shape)

team_games full: (22396, 39)


In [4]:
# pivot back to match level
team_cols = []
for w in [5, 10]:
    team_cols += [f'cf_l{w}', f'ca_l{w}',
                  f'attack_shrunk_l{w}', f'defense_shrunk_l{w}',
                  f'attack_above_l{w}',  f'defense_above_l{w}']
team_cols += ['cf_ewm_hl5', 'cf_ewm_hl10', 'ca_ewm_hl5', 'ca_ewm_hl10',
              'cf_adj_l10', 'ca_adj_l10']
venue_cols = ['cf_venue_l5', 'cf_venue_l10', 'ca_venue_l5', 'ca_venue_l10']

home_v = {f'cf_venue_l{w}': f'home_cf_at_home_l{w}' for w in [5,10]} | \
         {f'ca_venue_l{w}': f'home_ca_at_home_l{w}' for w in [5,10]}
away_v = {f'cf_venue_l{w}': f'away_cf_at_away_l{w}' for w in [5,10]} | \
         {f'ca_venue_l{w}': f'away_ca_at_away_l{w}' for w in [5,10]}

home_feats = (team_games[team_games['is_home'] == 1][['match_id'] + team_cols + venue_cols]
              .rename(columns={c: f'home_{c}' for c in team_cols}).rename(columns=home_v))
away_feats = (team_games[team_games['is_home'] == 0][['match_id'] + team_cols + venue_cols]
              .rename(columns={c: f'away_{c}' for c in team_cols}).rename(columns=away_v))

features = (model_data
            .merge(home_feats, on='match_id', how='left')
            .merge(away_feats, on='match_id', how='left')
            .merge(hist[['match_id'] + comp_cols], on='match_id', how='left'))

for w in [5, 10]:
    features[f'home_expected_l{w}'] = (features['comp_home_corners_prev']
                                       + features[f'home_attack_above_l{w}']
                                       + features[f'away_defense_above_l{w}'])
    features[f'away_expected_l{w}'] = (features['comp_away_corners_prev']
                                       + features[f'away_attack_above_l{w}']
                                       + features[f'home_defense_above_l{w}'])
    features[f'attack_edge_home_l{w}']       = features[f'home_cf_l{w}']         - features[f'away_ca_l{w}']
    features[f'attack_edge_away_l{w}']       = features[f'away_cf_l{w}']         - features[f'home_ca_l{w}']
    features[f'exp_total_l{w}']              = features[f'home_cf_l{w}']         + features[f'away_cf_l{w}']
    features[f'venue_attack_edge_home_l{w}'] = features[f'home_cf_at_home_l{w}'] - features[f'away_ca_at_away_l{w}']
    features[f'venue_attack_edge_away_l{w}'] = features[f'away_cf_at_away_l{w}'] - features[f'home_ca_at_home_l{w}']

features['adj_edge_home']  = features['home_cf_adj_l10'] - features['away_ca_adj_l10']
features['adj_edge_away']  = features['away_cf_adj_l10'] - features['home_ca_adj_l10']
features['ewm_edge_home']  = features['home_cf_ewm_hl5'] - features['away_ca_ewm_hl5']
features['ewm_edge_away']  = features['away_cf_ewm_hl5'] - features['home_ca_ewm_hl5']
features['ewm_exp_total']  = features['home_cf_ewm_hl5'] + features['away_cf_ewm_hl5']

print('features:', features.shape)

features: (11198, 81)


## NaN fill, train/val split, train baseline LightGBM

Chronological 80/20 split. L1 objective + early stopping on val so reported MAE
is mildly optimistic — fine for *relative* family ranking.

In [5]:
# fill NaN with sensible league-aware defaults
gh, ga = model_data['home_corners'].mean(), model_data['away_corners'].mean()
features[comp_cols] = features[comp_cols].fillna({'comp_home_corners_prev': gh, 'comp_away_corners_prev': ga,
                                                   'comp_total_corners_prev': gh + ga,
                                                   'comp_home_adv_prev': gh - ga})
for w in [5, 10]:
    features[f'home_cf_l{w}'] = features[f'home_cf_l{w}'].fillna(features['comp_home_corners_prev'])
    features[f'home_ca_l{w}'] = features[f'home_ca_l{w}'].fillna(features['comp_away_corners_prev'])
    features[f'away_cf_l{w}'] = features[f'away_cf_l{w}'].fillna(features['comp_away_corners_prev'])
    features[f'away_ca_l{w}'] = features[f'away_ca_l{w}'].fillna(features['comp_home_corners_prev'])
    features[f'home_cf_at_home_l{w}'] = features[f'home_cf_at_home_l{w}'].fillna(features['comp_home_corners_prev'])
    features[f'home_ca_at_home_l{w}'] = features[f'home_ca_at_home_l{w}'].fillna(features['comp_away_corners_prev'])
    features[f'away_cf_at_away_l{w}'] = features[f'away_cf_at_away_l{w}'].fillna(features['comp_away_corners_prev'])
    features[f'away_ca_at_away_l{w}'] = features[f'away_ca_at_away_l{w}'].fillna(features['comp_home_corners_prev'])
    features[f'home_attack_shrunk_l{w}']  = features[f'home_attack_shrunk_l{w}'].fillna(gh)
    features[f'home_defense_shrunk_l{w}'] = features[f'home_defense_shrunk_l{w}'].fillna(ga)
    features[f'away_attack_shrunk_l{w}']  = features[f'away_attack_shrunk_l{w}'].fillna(ga)
    features[f'away_defense_shrunk_l{w}'] = features[f'away_defense_shrunk_l{w}'].fillna(gh)
    for c in [f'home_attack_above_l{w}', f'home_defense_above_l{w}',
              f'away_attack_above_l{w}', f'away_defense_above_l{w}']:
        features[c] = features[c].fillna(0)
    features[f'home_expected_l{w}']           = features[f'home_expected_l{w}'].fillna(gh)
    features[f'away_expected_l{w}']           = features[f'away_expected_l{w}'].fillna(ga)
    features[f'attack_edge_home_l{w}']        = features[f'home_cf_l{w}']         - features[f'away_ca_l{w}']
    features[f'attack_edge_away_l{w}']        = features[f'away_cf_l{w}']         - features[f'home_ca_l{w}']
    features[f'exp_total_l{w}']               = features[f'home_cf_l{w}']         + features[f'away_cf_l{w}']
    features[f'venue_attack_edge_home_l{w}']  = features[f'home_cf_at_home_l{w}'] - features[f'away_ca_at_away_l{w}']
    features[f'venue_attack_edge_away_l{w}']  = features[f'away_cf_at_away_l{w}'] - features[f'home_ca_at_home_l{w}']
for hl in [5, 10]:
    features[f'home_cf_ewm_hl{hl}'] = features[f'home_cf_ewm_hl{hl}'].fillna(features['comp_home_corners_prev'])
    features[f'home_ca_ewm_hl{hl}'] = features[f'home_ca_ewm_hl{hl}'].fillna(features['comp_away_corners_prev'])
    features[f'away_cf_ewm_hl{hl}'] = features[f'away_cf_ewm_hl{hl}'].fillna(features['comp_away_corners_prev'])
    features[f'away_ca_ewm_hl{hl}'] = features[f'away_ca_ewm_hl{hl}'].fillna(features['comp_home_corners_prev'])
for c in ['home_cf_adj_l10', 'home_ca_adj_l10', 'away_cf_adj_l10', 'away_ca_adj_l10']:
    features[c] = features[c].fillna(0)
features['adj_edge_home']  = features['home_cf_adj_l10'] - features['away_ca_adj_l10']
features['adj_edge_away']  = features['away_cf_adj_l10'] - features['home_ca_adj_l10']
features['ewm_edge_home']  = features['home_cf_ewm_hl5'] - features['away_ca_ewm_hl5']
features['ewm_edge_away']  = features['away_cf_ewm_hl5'] - features['home_ca_ewm_hl5']
features['ewm_exp_total']  = features['home_cf_ewm_hl5'] + features['away_cf_ewm_hl5']

target_cols  = ['home_corners', 'away_corners']
exclude      = ['match_id', 'date_time', 'iw',
                'home_team_id', 'away_team_id',
                'home_ft_score', 'away_ft_score', 'corner_diff_home_minus_away',
                *target_cols, 'total_corners']
feature_cols = [c for c in features.columns if c not in exclude]
categorical_cols = ['competition_id', 'season_id']
for c in categorical_cols:
    features[c] = features[c].astype('category')

cutoff = features['date_time'].quantile(0.8)
train_feat = features[features['date_time'] <  cutoff].copy()
val_feat   = features[features['date_time'] >= cutoff].copy()
X_train, X_val = train_feat[feature_cols], val_feat[feature_cols]
y_train_h, y_val_h = train_feat['home_corners'], val_feat['home_corners']
y_train_a, y_val_a = train_feat['away_corners'], val_feat['away_corners']
sw = train_feat['iw'].fillna(1) if 'iw' in train_feat.columns else None

print(f'{len(feature_cols)} features | train: {X_train.shape} | val: {X_val.shape}')

70 features | train: (8956, 70) | val: (2242, 70)


In [6]:
kw = dict(objective='regression_l1', n_estimators=3000, learning_rate=0.03,
          num_leaves=31, min_child_samples=50,
          subsample=0.9, colsample_bytree=0.9, random_state=42, verbosity=-1)

lgb_h = LGBMRegressor(**kw)
lgb_a = LGBMRegressor(**kw)
lgb_h.fit(X_train, y_train_h, sample_weight=sw,
          eval_set=[(X_val, y_val_h)], callbacks=[early_stopping(100, verbose=False)],
          categorical_feature=categorical_cols)
lgb_a.fit(X_train, y_train_a, sample_weight=sw,
          eval_set=[(X_val, y_val_a)], callbacks=[early_stopping(100, verbose=False)],
          categorical_feature=categorical_cols)

base_h = mean_absolute_error(y_val_h, lgb_h.predict(X_val))
base_a = mean_absolute_error(y_val_a, lgb_a.predict(X_val))
print(f'baseline MAE  home: {base_h:.4f}  away: {base_a:.4f}')

baseline MAE  home: 2.2907  away: 1.9817


## Family definitions and importance helpers

In [7]:
groups = {
    'mixed_roll':  [c for c in feature_cols if c in
                    ['home_cf_l5','home_cf_l10','home_ca_l5','home_ca_l10',
                     'away_cf_l5','away_cf_l10','away_ca_l5','away_ca_l10']],
    'venue_split': [c for c in feature_cols if 'at_home' in c or 'at_away' in c],
    'ewm':         [c for c in feature_cols if 'ewm' in c],
    'adj':         [c for c in feature_cols if 'adj' in c],
    'shrunk':      [c for c in feature_cols if 'shrunk' in c],
    'above_norm':  [c for c in feature_cols if 'above' in c],
    'expected':    [c for c in feature_cols if 'expected' in c],
    'edges_mixed': [c for c in feature_cols if c.startswith('attack_edge') or c.startswith('exp_total')],
    'edges_venue': [c for c in feature_cols if c.startswith('venue_attack_edge')],
    'comp_priors': [c for c in feature_cols if c.startswith('comp_')],
    'cat':         categorical_cols + ['gameweek'],
}
for g, cols in groups.items():
    print(f'  {g:14s} {len(cols):3d}')

  mixed_roll       8
  venue_split      8
  ewm             11
  adj              6
  shrunk           8
  above_norm       8
  expected         4
  edges_mixed      6
  edges_venue      4
  comp_priors      4
  cat              3


In [8]:
rng = np.random.default_rng(0)

def perm_cols(X, cols, idx):
    Xp = X.copy()
    shuf = X[cols].iloc[idx].reset_index(drop=True)
    shuf.index = Xp.index
    for c in cols:
        Xp[c] = shuf[c]
    return Xp

def grouped_perm(model, X, y, base, n_repeats=5):
    rows = []
    for name, cols in groups.items():
        deltas = []
        for _ in range(n_repeats):
            idx = rng.permutation(len(X))
            deltas.append(mean_absolute_error(y, model.predict(perm_cols(X, cols, idx))) - base)
        rows.append({'group': name, 'n_cols': len(cols),
                     'deltaMAE_mean': np.mean(deltas),
                     'deltaMAE_std':  np.std(deltas)})
    return pd.DataFrame(rows).sort_values('deltaMAE_mean', ascending=False)

def single_perm(model, X, y, base, n_repeats=3):
    rows = []
    for c in feature_cols:
        deltas = []
        for _ in range(n_repeats):
            idx = rng.permutation(len(X))
            deltas.append(mean_absolute_error(y, model.predict(perm_cols(X, [c], idx))) - base)
        rows.append({'feature': c, 'deltaMAE': np.mean(deltas)})
    return pd.DataFrame(rows).sort_values('deltaMAE', ascending=False)

def builtin_gain(model, side):
    return pd.DataFrame({'feature': feature_cols,
                         f'{side}_gain': model.booster_.feature_importance(importance_type='gain')})

## Built-in gain (training-side reference)

In [9]:
gain = (builtin_gain(lgb_h, 'home')
        .merge(builtin_gain(lgb_a, 'away'), on='feature')
        .assign(total_gain=lambda d: d['home_gain'] + d['away_gain'])
        .sort_values('total_gain', ascending=False))
display(gain.head(20).round(0))

,feature,home_gain,away_gain,total_gain
1,season_id,5030.0,5206.0,10236.0
40,away_ca_ewm_hl10,1683.0,1354.0,3037.0
16,home_cf_ewm_hl10,2163.0,832.0,2995.0
18,home_ca_ewm_hl10,1038.0,1928.0,2967.0
59,away_expected_l10,762.0,1663.0,2425.0
35,away_attack_above_l10,1040.0,1350.0,2390.0
41,away_cf_adj_l10,981.0,1260.0,2241.0
38,away_cf_ewm_hl10,1025.0,1079.0,2104.0
34,away_defense_shrunk_l10,1266.0,788.0,2054.0
24,home_ca_at_home_l10,1061.0,965.0,2026.0


## Grouped permutation — the deciding metric

ΔMAE > 0 means shuffling the family hurts val performance. Larger = more
signal that *no other family covers*. Negative or near-zero = redundant.

In [10]:
print('=== HOME ===')
gp_h = grouped_perm(lgb_h, X_val, y_val_h, base_h)
display(gp_h.round(4))

print('=== AWAY ===')
gp_a = grouped_perm(lgb_a, X_val, y_val_a, base_a)
display(gp_a.round(4))

=== HOME ===


,group,n_cols,deltaMAE_mean,deltaMAE_std
2,ewm,11,0.0694,0.0103
10,cat,3,0.0180,0.0039
1,venue_split,8,0.0128,0.0039
6,expected,4,0.0081,0.0024
5,above_norm,8,0.0047,0.0018
4,shrunk,8,0.0037,0.0008
7,edges_mixed,6,0.0014,0.0015
9,comp_priors,4,0.0014,0.0003
0,mixed_roll,8,0.0009,0.0007
3,adj,6,-0.0003,0.0008


=== AWAY ===


,group,n_cols,deltaMAE_mean,deltaMAE_std
2,ewm,11,0.0605,0.0038
5,above_norm,8,0.0120,0.0020
3,adj,6,0.0074,0.0025
6,expected,4,0.0061,0.0010
10,cat,3,0.0046,0.0007
8,edges_venue,4,0.0020,0.0015
1,venue_split,8,0.0017,0.0017
4,shrunk,8,0.0005,0.0009
7,edges_mixed,6,-0.0010,0.0011
0,mixed_roll,8,-0.0012,0.0023


## Single-feature permutation — top contributors

Useful for seeing which individual features inside a kept family are doing the work.
Suffers from collinearity: highly correlated features may all show small Δ even
when the *family* is essential, which is why the grouped score is the decision-maker.

In [11]:
sp_h = single_perm(lgb_h, X_val, y_val_h, base_h)
sp_a = single_perm(lgb_a, X_val, y_val_a, base_a)

print('=== HOME top 15 ===')
display(sp_h.head(15).round(4))
print('=== AWAY top 15 ===')
display(sp_a.head(15).round(4))

=== HOME top 15 ===


,feature,deltaMAE
16,home_cf_ewm_hl10,0.0284
1,season_id,0.0202
40,away_ca_ewm_hl10,0.0183
24,home_ca_at_home_l10,0.0053
34,away_defense_shrunk_l10,0.0046
58,home_expected_l10,0.0042
38,away_cf_ewm_hl10,0.0037
18,home_ca_ewm_hl10,0.0023
36,away_defense_above_l10,0.0020
27,away_attack_shrunk_l5,0.0019


=== AWAY top 15 ===


,feature,deltaMAE
18,home_ca_ewm_hl10,0.0203
35,away_attack_above_l10,0.0090
40,away_ca_ewm_hl10,0.0082
17,home_ca_ewm_hl5,0.0069
38,away_cf_ewm_hl10,0.0062
59,away_expected_l10,0.0055
1,season_id,0.0053
20,home_ca_adj_l10,0.0042
16,home_cf_ewm_hl10,0.0037
24,home_ca_at_home_l10,0.0030


## Decision

**Drop families** (grouped ΔMAE ≤ 0 on both targets):
- `mixed_roll` — replaced by EWM
- `shrunk` — covered by `above_norm`
- `edges_mixed` — trees can derive these from raw rolling
- `comp_priors` — `season_id` (categorical) + `expected` columns absorb its signal

**Keep families** (grouped ΔMAE > 0):
- `ewm` (dominant: ΔMAE ≈ +0.06 home, +0.05 away)
- `cat` (`competition_id`, `season_id`, `gameweek`)
- `expected`, `venue_split`, `above_norm`
- `adj`, `edges_venue` (mild but consistent)

Result: 70 → 44 features. Q1.ipynb runs on the slim set with no MAE regression.